# 🧊 TRELLIS 3D – تحويل الصورة إلى مجسم (آلي بالكامل)
**قبل التشغيل:** تأكد من تفعيل GPU (Runtime → Change runtime type → T4 GPU).
لن يطلب منك أي رفع يدوي – كل شيء سيحدث تلقائياً.

In [ ]:
# =============================================
# TRELLIS 3D – خلية واحدة (تحميل تلقائي متعدد المحاولات)
# =============================================
import os, sys, subprocess, zipfile, io, requests, tempfile, traceback, importlib.metadata
import torch, numpy as np
from PIL import Image
import gradio as gr

# 1. فحص GPU
print("🔍 فحص GPU...")
if not torch.cuda.is_available():
    print("❌ لا يوجد GPU. الرجاء: Runtime → Change runtime type → GPU")
    exit(0)

!nvidia-smi
gpu_name = subprocess.getoutput("nvidia-smi --query-gpu=name --format=csv,noheader")
print(f"✅ GPU: {gpu_name}")

SUPPORTED_GPU = ["A100", "A6000", "3090", "4090", "L4", "L40", "H100"]
USE_FLASH_ATTN = any(g in gpu_name for g in SUPPORTED_GPU)
print("✨ Flash Attention 2 مدعوم" if USE_FLASH_ATTN else "ℹ️  يستخدم SDPA.")

# 2. تثبيت الحزم الأساسية
def is_installed(pkg):
    try:
        importlib.metadata.version(pkg)
        return True
    except importlib.metadata.PackageNotFoundError:
        return False

print("\n🔍 فحص الحزم...")
base_pkgs = ["trimesh", "pygltflib", "viser", "omegaconf", "opencv-python", "imageio", "gradio", "huggingface_hub"]
for pkg in base_pkgs:
    if not is_installed(pkg):
        !pip install -q {pkg}
        print(f"✔️ {pkg}")
    else:
        print(f"✅ {pkg}")

# spconv (بدون تحديد إصدار CUDA)
if not is_installed("spconv"):
    print("⏳ تثبيت spconv...")
    try:
        !pip install -q spconv
    except:
        !pip install -q spconv-cu118
    print("✔️ spconv")
else:
    print("✅ spconv")

# Flash Attention (اختياري)
if USE_FLASH_ATTN and not is_installed("flash-attn"):
    print("⚡ تثبيت Flash Attention 2...")
    !pip install -q ninja packaging
    !pip install flash-attn --no-build-isolation 2>&1 | tail -3
    import flash_attn
    print("✅ flash-attn")

# 3. تحميل مكتبة TRELLIS تلقائياً (باستخدام عدة روابط بديلة)
TARGET_DIR = "/content/TRELLIS-main"
if not os.path.isdir(TARGET_DIR):
    print("\n⬇️  تحميل كود TRELLIS...")
    success = False
    # قائمة الروابط البديلة (مرتبة حسب الأفضلية)
    urls = [
        "https://codeload.github.com/JeffreyXiang/TRELLIS/zip/main",
        "https://api.github.com/repos/JeffreyXiang/TRELLIS/zipball/main",
        "https://github.com/JeffreyXiang/TRELLIS/archive/main.zip"
    ]

    for url in urls:
        print(f"   محاولة: {url}")
        try:
            # الطريقة 1: requests
            r = requests.get(url, timeout=30, allow_redirects=True)
            r.raise_for_status()
            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                z.extractall("/content")
            success = True
            print("   ✅ تم التحميل بنجاح.")
            break
        except Exception as e:
            print(f"   ❌ فشل (requests): {e}")
            # الطريقة 2: wget (أحياناً يتجاوز بعض القيود)
            try:
                !wget -q -O /tmp/trellis.zip {url}
                if os.path.exists("/tmp/trellis.zip"):
                    with zipfile.ZipFile("/tmp/trellis.zip") as z:
                        z.extractall("/content")
                    success = True
                    print("   ✅ تم التحميل عبر wget.")
                    break
            except:
                pass
            # الطريقة 3: curl (أخيراً)
            try:
                !curl -L -o /tmp/trellis_curl.zip {url}
                if os.path.exists("/tmp/trellis_curl.zip"):
                    with zipfile.ZipFile("/tmp/trellis_curl.zip") as z:
                        z.extractall("/content")
                    success = True
                    print("   ✅ تم التحميل عبر curl.")
                    break
            except:
                pass

    if not success:
        raise RuntimeError("❌ فشل تحميل كود TRELLIS بجميع الطرق. تحقق من اتصالك بالإنترنت.")

if not os.path.isdir(TARGET_DIR):
    raise FileNotFoundError("مجلد TRELLIS-main غير موجود. فشل التحميل.")

sys.path.insert(0, TARGET_DIR)
import trellis
print("✅ استيراد trellis ناجح.")

# إعدادات الانتباه
if USE_FLASH_ATTN:
    try:
        import flash_attn
        attn_impl = "flash_attention_2"
        print("🔧 Flash Attention 2.")
    except:
        attn_impl = "sdpa"
        print("🔧 SDPA.")
else:
    attn_impl = "sdpa"
    print("🔧 SDPA.")

# 4. تحميل النموذج
from trellis.pipelines import TrellisImageTo3DPipeline
try:
    pipeline = TrellisImageTo3DPipeline.from_pretrained(
        "JeffreyXiang/TRELLIS-image-large",
        torch_dtype=torch.float16,
        attn_implementation=attn_impl
    ).to("cuda")
    print("✅ نموذج TRELLIS محمّل.")
except Exception as e:
    print(f"⚠️ خطأ: {e}\nمحاولة بدون attn_implementation...")
    pipeline = TrellisImageTo3DPipeline.from_pretrained(
        "JeffreyXiang/TRELLIS-image-large",
        torch_dtype=torch.float16
    ).to("cuda")
    print("✅ نموذج TRELLIS محمّل.")

# 5. دالة التوليد
def generate_3d(image_input, seed, cfg_scale):
    if image_input is None:
        raise gr.Error("يرجى رفع صورة.")
    image = Image.fromarray(image_input).convert("RGB") if isinstance(image_input, np.ndarray) else image_input.convert("RGB")
    try:
        try:
            outputs = pipeline.run(image, seed=int(seed), cfg_scale=float(cfg_scale))
        except TypeError:
            outputs = pipeline.run(image, seed=int(seed))
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"فشل التوليد: {e}")
    mesh = outputs.get('mesh')
    if mesh is None:
        raise gr.Error("لم يتم توليد شبكة.")
    tmpdir = tempfile.mkdtemp()
    glb_path = os.path.join(tmpdir, "model.glb")
    mesh.export(glb_path)
    return glb_path, glb_path

# 6. واجهة Gradio
with gr.Blocks(title="TRELLIS 3D", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🧊 TRELLIS – تحويل الصورة إلى مجسم ثلاثي الأبعاد
    أرفع صورة واضغط **توليد**. يمكنك تدوير المجسم بالماوس.
    """)
    with gr.Row():
        with gr.Column():
            image_input = gr.Image(label="الصورة المدخلة", type="pil", image_mode="RGB")
            with gr.Accordion("⚙️ إعدادات", open=False):
                seed = gr.Number(value=42, label="Seed", precision=0)
                cfg_scale = gr.Slider(1.0, 20.0, 7.5, step=0.5, label="CFG Scale")
            generate_btn = gr.Button("🚀 توليد", variant="primary")
        with gr.Column():
            model_output = gr.Model3D(label="المجسم")
            download_btn = gr.File(label="تحميل GLB")
    generate_btn.click(fn=generate_3d, inputs=[image_input, seed, cfg_scale], outputs=[model_output, download_btn])

print("\n🌐 تشغيل واجهة Gradio...")
demo.queue(max_size=3).launch(share=True, server_port=7860, show_error=True)